In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

BASE_PATH1 = "/content/drive/MyDrive/research"
BASE_PATH2="/content/drive/MyDrive/research/anca/preprocessing"
INPUT_STUDENTINFO = os.path.join(BASE_PATH1, "studentInfo_4_partialgrades_again.csv")
ORIG_STUDENT_ASSESSMENT = os.path.join(BASE_PATH2, "original_studentAssessment.csv")
ORIG_ASSESSMENTS = os.path.join(BASE_PATH2, "original_assessments.csv")
OUTPUT_BINNED = os.path.join(BASE_PATH1, "dataset1.csv")


def build_day_bins(max_day: int = 260):
    bins = []
    start = 0
    end = min(26, max_day)
    bins.append((start, end))
    while end < max_day:
        start = end + 1
        end = min(start + 25, max_day)
        bins.append((start, end))
    return bins


def date_to_bin_id(date_series: pd.Series, max_day: int = 260) -> pd.Series:
    d = date_series.clip(lower=0, upper=max_day).astype(int)
    b = np.where(d <= 26, 0, 1 + ((d - 27) // 26))
    return pd.Series(b, index=date_series.index, dtype=int)


def main():

    student_df = pd.read_csv(INPUT_STUDENTINFO)
    assess_df = pd.read_csv(ORIG_ASSESSMENTS)
    stud_assess_df = pd.read_csv(ORIG_STUDENT_ASSESSMENT)

    assess_df = assess_df[assess_df["assessment_type"].isin(["TMA", "CMA"])].copy()

    assess_df["weight"] = pd.to_numeric(assess_df["weight"], errors="coerce").fillna(0.0)
    assess_df["date"] = pd.to_numeric(assess_df["date"], errors="coerce").fillna(0).astype(int)
    stud_assess_df["score"] = pd.to_numeric(stud_assess_df["score"], errors="coerce").fillna(0.0)

    max_day = 260
    bins = build_day_bins(max_day=max_day)

    assess_df["bin_id"] = date_to_bin_id(assess_df["date"], max_day=max_day)

    total_w = (
        assess_df.groupby(["code_module", "code_presentation"], as_index=False)["weight"]
        .sum()
        .rename(columns={"weight": "total_assess_weight"})
    )

    assigned_w_bin = (
        assess_df.groupby(["code_module", "code_presentation", "bin_id"], as_index=False)["weight"]
        .sum()
        .rename(columns={"weight": "assigned_weight"})
    )

    assigned_w_bin = assigned_w_bin.merge(
        total_w, on=["code_module", "code_presentation"], how="left"
    )

    # W_i = assigned fraction (0–1)
    assigned_w_bin["W"] = np.divide(
        assigned_w_bin["assigned_weight"],
        assigned_w_bin["total_assess_weight"],
        out=np.zeros_like(assigned_w_bin["assigned_weight"], dtype=float),
        where=assigned_w_bin["total_assess_weight"] > 0,
    ) * 100

    merged = stud_assess_df.merge(
        assess_df[["id_assessment", "code_module", "code_presentation", "weight", "bin_id"]],
        on="id_assessment",
        how="inner",
    )

    merged = (
        merged.groupby(
            ["id_student", "id_assessment", "code_module", "code_presentation", "weight", "bin_id"],
            as_index=False
        )["score"]
        .max()
    )

    merged["secured_points"] = merged["weight"] * (merged["score"] / 100.0)

    secured_bin = (
        merged.groupby(["id_student", "code_module", "code_presentation", "bin_id"], as_index=False)["secured_points"]
        .sum()
    )

    secured_bin = secured_bin.merge(total_w, on=["code_module", "code_presentation"], how="left")

    # secured ratio relative to total weight
    secured_bin["secured_ratio"] = np.divide(
        secured_bin["secured_points"],
        secured_bin["total_assess_weight"],
        out=np.zeros_like(secured_bin["secured_points"], dtype=float),
        where=secured_bin["total_assess_weight"] > 0,
    )

    assigned_wide = assigned_w_bin.pivot_table(
        index=["code_module", "code_presentation"],
        columns="bin_id",
        values="W",
        fill_value=0.0,
        aggfunc="first",
    )

    assigned_wide.columns = [f"W_{int(b)+1}" for b in assigned_wide.columns]
    assigned_wide = assigned_wide.reset_index()

    secured_wide = secured_bin.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="bin_id",
        values="secured_ratio",
        fill_value=0.0,
        aggfunc="first",
    )

    secured_wide.columns = [f"S_{int(b)+1}" for b in secured_wide.columns]
    secured_wide = secured_wide.reset_index()

    out = student_df.merge(
        assigned_wide, on=["code_module", "code_presentation"], how="left"
    ).merge(
        secured_wide, on=["id_student", "code_module", "code_presentation"], how="left"
    )

    g_cols_new = [c for c in out.columns if c.startswith("W_") or c.startswith("S_")]
    out[g_cols_new] = out[g_cols_new].fillna(0.0)

    old_g_cols = [c for c in student_df.columns if c.startswith("g_")]
    out = out.drop(columns=old_g_cols, errors="ignore")

    # Compute V_i = secured / assigned
    for i in range(1, 11):
        w_col = f"W_{i}"
        s_col = f"S_{i}"
        v_col = f"V_{i}"

        out[v_col] = np.divide(
            out[s_col],
            out[w_col],
            out=np.zeros(len(out)),
            where=out[w_col] > 0
        ) * 100

    # Drop temporary secured columns
    s_cols = [f"S_{i}" for i in range(1, 11)]
    out = out.drop(columns=s_cols)

    desired_pairs = []
    for i in range(1, 11):
        desired_pairs.extend([f"W_{i}", f"V_{i}"])

    for c in desired_pairs:
        if c not in out.columns:
            out[c] = 0.0

    base_cols = [c for c in out.columns if c not in desired_pairs]
    out = out[base_cols + desired_pairs]

    out.to_csv(OUTPUT_BINNED, index=False)

    print(f"Saved dataset: {OUTPUT_BINNED}")
    print(f"Shape: {out.shape}")
    print("New columns:", desired_pairs)


if __name__ == "__main__":
    main()

Saved dataset: /content/drive/MyDrive/research/dataset1.csv
Shape: (24615, 49)
New columns: ['W_1', 'V_1', 'W_2', 'V_2', 'W_3', 'V_3', 'W_4', 'V_4', 'W_5', 'V_5', 'W_6', 'V_6', 'W_7', 'V_7', 'W_8', 'V_8', 'W_9', 'V_9', 'W_10', 'V_10']


In [ ]:
import pandas as pd

# Load the dataset
dataset1 = pd.read_csv("/content/drive/MyDrive/research/dataset1.csv")

# Check for non-zero values in W_10 and V_10
non_zero_W10 = dataset1['W_10'].ne(0).any()
non_zero_V10 = dataset1['V_10'].ne(0).any()

print(f"W_10 has values other than 0: {non_zero_W10}")
print(f"V_10 has values other than 0: {non_zero_V10}")

rows_with_nonzero = dataset1[(dataset1['W_10'] != 0) | (dataset1['V_10'] != 0)]
print("Rows with non-zero W_10 or V_10:")
print(rows_with_nonzero)

W_10 has values other than 0: False
V_10 has values other than 0: False
Rows with non-zero W_10 or V_10:
Empty DataFrame
Columns: [code_module, code_presentation, id_student, gender, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability, final_result, assessment_grade, exam_grade, course_category, final_grade, region_East Anglian Region, region_East Midlands Region, region_Ireland, region_London Region, region_North Region, region_North Western Region, region_Scotland, region_South East Region, region_South Region, region_South West Region, region_Wales, region_West Midlands Region, region_Yorkshire Region, VLE_normalized, W_1, V_1, W_2, V_2, W_3, V_3, W_4, V_4, W_5, V_5, W_6, V_6, W_7, V_7, W_8, V_8, W_9, V_9, W_10, V_10]
Index: []

[0 rows x 49 columns]


In [ ]:
import pandas as pd

dataset_path = "/content/drive/MyDrive/research/dataset1.csv"
df = pd.read_csv(dataset_path)

# List of W columns
w_cols = [f"W_{i}" for i in range(1, 11)]

# Compute the total weight per row
df["W_total"] = df[w_cols].sum(axis=1)

# Check if the total is exactly 100
df["W_total_check"] = df["W_total"] == 100

# Print summary
num_invalid = (~df["W_total_check"]).sum()
print(f"Number of rows where sum of W_1 to W_10 is NOT 100: {num_invalid}")

if num_invalid > 0:
    print("Example rows with invalid total:")
    print(df.loc[~df["W_total_check"], w_cols + ["W_total"]].head())

print("\nSummary of total weights (W_total):")
print(df["W_total"].describe())

Number of rows where sum of W_1 to W_10 is NOT 100: 0

Summary of total weights (W_total):
count    24615.0
mean       100.0
std          0.0
min        100.0
25%        100.0
50%        100.0
75%        100.0
max        100.0
Name: W_total, dtype: float64


In [ ]:
import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/research"
INPUT_FILE = os.path.join(BASE_PATH, "dataset1.csv")

# Load dataset
df = pd.read_csv(INPUT_FILE)

# Keep only rows where exam_grade is not zero
df_exam = df[df['exam_grade'].notna() & (df['exam_grade'] != 0)]

# Count unique combinations of code_module and code_presentation
unique_combinations = df_exam[['code_module', 'code_presentation']].drop_duplicates()
num_unique = len(unique_combinations)

print(f"Number of unique code_module + code_presentation combinations with exam_grade > 0: {num_unique}")
print("Unique combinations:")
print(unique_combinations)

Number of unique code_module + code_presentation combinations with exam_grade > 0: 6
Unique combinations:
      code_module code_presentation
6866          CCC             2014B
8505          CCC             2014J
10717         DDD             2013B
11807         DDD             2013J
13377         DDD             2014B
14364         DDD             2014J
